## Task 1: Load Documents

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

# Initialize Pdf loader
loader = PyPDFLoader('physical.pdf')

# loading PDF
docs = loader.load()

print("Number of docs:", len(docs))

print(docs[0].page_content)

## Task 2: Text Splitting

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initializing RecursiveCharacterTextSplitter with chunk_size and chunk_overlap
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Splitting doc into chunks
chunks = splitter.split_documents(docs)

print("Number of chunks:", len(chunks))

## Task 3: Create Embeddings

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

# Loading embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Creating embedding of chunk from chunks
vectors = embeddings.embed_documents(
    [chunk.page_content for chunk in chunks]
)

print("Number of embeddings:", len(vectors))
print("Embedding dimensions:", len(vectors[0]))

## Task 4: Store Embeddings in Vector Store

In [ ]:
from langchain_chroma import Chroma

# Storing in vector store
vector_store = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory="./chroma_db")

print("Documents stored in ChromaDB successfully!")

# Creating retriever from vector store
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

## Task 5: RAG Prompt Template

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# Creating Chat Prompt with system instruction, MessagesPlaceholder for history and user input
prompt = ChatPromptTemplate.from_messages([
    ('system', '''You are a helpful AI assistant that answers questions
    using the provided document context.
    Use conversation history to understand follow-up questions.
    Answer only using the retrieved context.
    If the answer is not present in the context, say
    "I don't know based on the provided documents."   

    Retrieved context:
    {context} 
    '''),
    MessagesPlaceholder(variable_name='history'), 
    ('human', '{input}')
])

## Task 6: Build RAG Chain

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# Initialize ChatGroq Model
llm = ChatGroq(model="openai/gpt-oss-120b")

# String parser
parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Combine retrieved documents with the prompt
ques_ans_chain = create_stuff_documents_chain(llm, prompt)

# Create retrieval chain using your existing retriever
rag_chain = create_retrieval_chain(retriever, ques_ans_chain)

chat_history = []

response = rag_chain.invoke({
    'history': chat_history,
    'input': 'What is Corona Virus'
})

print(response['answer'])


## Task 7: Maintain Message History

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = []

def chat(input):
    response = rag_chain.invoke({
    'history': chat_history,
    'input': input
    })

    answer = response['answer']

    chat_history.append(HumanMessage(content=input))
    chat_history.append(AIMessage(content=answer))

    return answer

print(chat("What is Corona Virus?"))
print(chat("What are its common symptoms?"))
print(chat("Who is Mahatma Gandhi"))

## Task 8: Trimming Chat History

In [ ]:
from langchain_core.messages import trim_messages
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

trimmer = trim_messages(token_counter=llm, max_tokens=2000, strategy='last', include_system=True, allow_partial=False, start_on='human')

def chat(question):
    trimmed_history = trimmer.invoke(chat_history)

    response = rag_chain.invoke({
        "input": question,
        "history": trimmed_history
    })

    answer = response["answer"]

    # Keep the full history for future trimming
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

    return answer


print(chat("What is Corona Virus?"))
print(chat("What are its common symptoms?"))
print(chat("What is Physical Education"))

print("\nTotal messages stored:", len(chat_history))
print("Messages sent after trimming:", len(trimmer.invoke(chat_history)))

## Task 9: Multi-Turn Q&A Testing

In [ ]:
chat_history = []

questions = [
    'What is Corona Virus?',
    'What are its common symptoms?',
    'How does it spread?',
    'How can we prevent it?',
]

for i, ques in enumerate(questions, start=1):
    answer = chat(ques)
    print('\nQ', i, ques)
    print('A', i, answer)

## Task 10: Build Final Chatbot

In [ ]:
# Refer aap.py for steamlit app

## Task 11: Observations & Insights